In [1]:
# STEP 9: Install and import required libraries

%pip install -q ultralytics pyyaml

import torch
import ultralytics
import yaml
import os

print("PyTorch version:", torch.__version__)
print("Ultralytics version:", ultralytics.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("No NVIDIA GPU detected. This is okay for now.")

Note: you may need to restart the kernel to use updated packages.
PyTorch version: 2.15.0
Ultralytics version: 8.4.208
CUDA available: False
No NVIDIA GPU detected. This is okay for now.


In [2]:
# STEP 10: Kaggle dataset configuration

from pathlib import Path
import yaml

# Find the dataset after it is attached to Kaggle
input_dir = Path("/kaggle/input")

candidates = list(input_dir.glob("**/randomized_dataset"))

if candidates:
    dataset_root = candidates[0]
    print("Dataset found:", dataset_root)
else:
    print("Dataset not found yet.")
    print("Attach crackathon-data using Kaggle's Add Input option.")

# Prepare YOLO configuration
if candidates:
    config = {
        "path": str(dataset_root),
        "train": "train/images",
        "val": "val/images",
        "nc": 5,
        "names": [
            "Longitudinal Crack",
            "Transverse Crack",
            "Alligator Crack",
            "Other Corruption",
            "Pothole"
        ]
    }

    yaml_path = Path("/kaggle/working/road_damage_data.yaml")

    with open(yaml_path, "w") as f:
        yaml.safe_dump(config, f, sort_keys=False)

    print("Kaggle YAML created:", yaml_path)

Dataset not found yet.
Attach crackathon-data using Kaggle's Add Input option.


In [ ]:
# STEP 11: YOLOv8 baseline training

from ultralytics import YOLO
from pathlib import Path
import torch

# Check that the Kaggle dataset configuration exists
assert Path(yaml_path).exists(), "Dataset YAML file not found!"

# Check GPU availability
assert torch.cuda.is_available(), "Please enable GPU in Kaggle Settings!"

print("GPU:", torch.cuda.get_device_name(0))

# Load pretrained YOLOv8 Nano
model = YOLO("yolov8n.pt")

# Start baseline training
results = model.train(
    data=str(yaml_path),
    epochs=20,
    imgsz=640,
    batch=16,
    workers=2,
    device=0,
    project="/kaggle/working/road_damage_experiments",
    name="baseline_yolov8n",
    pretrained=True,
    plots=True,
    save=True
)

print("Baseline training completed!")